In [1]:
using Pkg
Pkg.instantiate()
Pkg.update()

    Updating registry at `~/.julia/registries/General.toml`
    Updating git-repo `https://github.com/euriqa-brassboard/MSSim.jl.git`
     Project No packages added to or removed from `~/projects/yyc-data/euriqa/calculations/rydberg_czs/Project.toml`
    Manifest No packages added to or removed from `~/projects/yyc-data/euriqa/calculations/rydberg_czs/Manifest.toml`
        Info We haven't cleaned this depot up for a bit, running Pkg.gc()...
      Active manifest files: 9 found
      Active artifact files: 1 found
      Active scratchspaces: 0 found
     Deleted no artifacts, repos, packages or scratchspaces


In [2]:
include("sqrt_cz.jl")

opt_n! (generic function with 1 method)

In [3]:
using NPZ

In [4]:
Ω = 2π * 3
nseg = 30
nsubsample = 30
t_gate = 0.8
t_ramp = 0.01
fm_limit = 2π * 10 * 1.5
opt = SplineOpt(Ω=Ω, nseg=nseg, nsubsample=nsubsample, t_gate=t_gate, t_ramp=t_ramp,
                lam_rob=0.0, lam_leak=0.0, lam_dark=0.0, fm_limit=fm_limit, maxtime=10, maxeval_pre=10000);
# opt = Opt(Ω=Ω, num_slices=num_slices, t_gate=t_gate,
#           lam_rob=0.1, lam_leak=1, lam_dark=1);
# optional keyword arguments:
# algorithm=:LD_CCSAQ, maxeval_pre=1000, maxtime=3, xtol=1e-7, minω=-2π * 10, maxω=2π * 10

In [5]:
best_obj, best_args = @time opt_n!(opt, 100; verbose=false, pre_threshold=0.05) # default verbosity is true

obj = 0.180378001396565
obj = 0.15786777336251612
obj = 0.14634175017362816
obj = 0.13491839800375738
obj = 0.001533267156360596
Round 20 done
Round 40 done
Round 60 done
Round 80 done
Round 100 done
  2.161542 seconds (530.66 k allocations: 24.372 MiB, 11.81% compilation time)


(0.001533267156360596, [11.778788077581904, 14.063527372667862, 15.366502211475192, 17.440217041995318, 16.693175850041204, 16.93381627497981, 14.411001899496837, 13.757656522321737, 16.414497508080142, 19.06988436503948  …  11.961562607707005, 9.562654267417567, 9.733595842666857, 10.934070933696455, 13.337689921482323, 15.126370047447447, 13.919487039957751, 11.27596034765188, 8.69957814651327, -8.879927381835945])

In [6]:
fm_rate_constraints(best_args, (), 1:opt.idx_max, opt.diff_limit)

0.14914967927862932

In [7]:
# More tries to refine the result.
for _ in 1:25
    best_obj, best_args = @time opt_n!(opt, 100; pre_threshold=0.05,
                                       verbose=false, best_obj=best_obj, best_args=best_args)
    if best_obj < 1e-4
        break
    end
end

Round 20 done
Round 40 done
Round 60 done
Round 80 done
Round 100 done
  2.104501 seconds (153.05 k allocations: 3.984 MiB, 5.76% gc time, 0.44% compilation time)
Round 20 done
Round 40 done
Round 60 done
Round 80 done
Round 100 done
  1.905746 seconds (145.71 k allocations: 3.727 MiB)
Round 20 done
Round 40 done
Round 60 done
Round 80 done
Round 100 done
  1.900626 seconds (145.59 k allocations: 3.723 MiB)
Round 20 done
Round 40 done
Round 60 done
Round 80 done
Round 100 done
  2.010389 seconds (154.11 k allocations: 3.940 MiB)
Round 20 done
obj = 0.0012707268473541866
Round 40 done
Round 60 done
obj = 0.0012613362219430213
Round 80 done
Round 100 done
  2.088035 seconds (159.97 k allocations: 4.094 MiB)
Round 20 done
Round 40 done
Round 60 done
Round 80 done
Round 100 done
  1.825078 seconds (139.46 k allocations: 3.568 MiB)
Round 20 done
Round 40 done
Round 60 done
obj = 0.0009499352864332788
Round 80 done
Round 100 done
  1.933516 seconds (147.76 k allocations: 3.783 MiB)
Round 20 

In [8]:
best_ϕs = fm_to_phase(opt, best_args)
println(best_ϕs)

[0.0, 0.010212344855115164, 0.020417746466995925, 0.030616346388722925, 0.04080842772645743, 0.050994415139441404, 0.06117487483999737, 0.07135051459352855, 0.08152218371851883, 0.09169087308653269, 0.10185771512221516, 0.11202398380329213, 0.12219109466057, 0.13236060477793576, 0.14253421279235723, 0.1527137588938827, 0.16290122482564104, 0.17309873388384198, 0.18330855091777573, 0.19353308232981323, 0.2037748760754061, 0.2140366216630864, 0.22432115015446705, 0.23463143416424143, 0.2449705878601838, 0.2553418669631487, 0.2657486687470717, 0.2761945320389689, 0.28668313721893696, 0.2972183062201528, 0.3078040025288748, 0.3184443311844415, 0.32914345504364884, 0.3399052598382543, 0.3507332704393558, 0.3616306508573899, 0.37260020424213164, 0.3836443728826956, 0.3947652382075357, 0.40596452078444434, 0.4172435803205531, 0.4286034156623329, 0.44004466479559334, 0.4515676048454832, 0.4631721520764908, 0.4748578618924428, 0.4866239288365048, 0.4984691865911826, 0.51039210797832, 0.52239080

In [9]:
npzwrite("sqrtcz_0.6us_3MHz.npz", Dict("phase_list"=>best_ϕs, "t_gate"=>t_gate, "t_ramp"=>t_ramp, "Omega"=>Ω, "Omegas"=>Vector(opt.cb.Ωs)))